# Pharmasense — Graph WaveNet: Adaptive Spatio-Temporal Forecasting
**Learns adaptive adjacency matrix from disease transmission data — not just geographic borders.**

| vs ST-GNN | Graph WaveNet advantage |
|---|---|
| Fixed border adjacency | Learnable adjacency: E₁·E₂ᵀ |
| GRU (sequential) | Dilated causal convolutions (parallelizable) |
| 64-week receptive field | 2^L × kernel_size receptive field |

Run ST-GNN (01) first, then this. Final inference = ensemble average of both.

## Before running
1. **Runtime → T4 GPU**
2. Run notebook 01 first and download `stgnn_best.pt`
3. **Runtime → Run All**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/03_train_graphwavenet.ipynb)

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print(f'GPU: {r.stdout.strip()}' if r.returncode==0 else 'No GPU — will be very slow.')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.2.0', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric', 'torch_scatter', 'torch_sparse', 'torch_cluster',
    '-f', 'https://data.pyg.org/whl/torch-2.2.0+cu118.html'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'numpy', 'scikit-learn', 'tqdm', 'matplotlib', 'seaborn'], check=True)
print('Dependencies installed.')

In [ ]:
GITHUB_RAW = 'https://raw.githubusercontent.com/abhi-dev99/agtechathon/main/ml/data'
DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
import os
from google.colab import files
REQUIRED_FILES = ['malaria_weekly.csv.gz', 'epiclim_clean.csv', 'district_index.json']
missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
if missing:
    print('GitHub repository is private. Please upload the following files from your local ml/data/ folder:\n')
    for f in missing:
        print(f'  - {f}')
    print('\n')
    uploaded = files.upload()
    missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
    if missing:
        raise FileNotFoundError(f'Still missing: {missing}')
print('All data files are present.')


In [ ]:
import json, pandas as pd, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

with open('/content/district_index.json') as f:
    graph_data = json.load(f)
node_list   = graph_data['nodes']
edge_list   = graph_data['edges']
N_NODES     = len(node_list)
name_to_idx = {n['name'].lower().strip(): i for i, n in enumerate(node_list)}

# Build adjacency matrix (sparse)
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
adj = torch.zeros(N_NODES, N_NODES)
adj[edge_index[0], edge_index[1]] = 1
adj = (adj + adj.t()).clamp(0, 1)  # symmetric
# Normalize
deg = adj.sum(dim=1, keepdim=True).clamp(min=1)
adj_norm = (adj / deg).to(DEVICE)
print(f'Graph: {N_NODES} nodes, {len(edge_list)} edges')

In [ ]:
# ── Load case data from CSVs ──────────────────────────────────────────────────
disease_names = ['malaria','dengue','cholera']
D = len(disease_names)

mal = pd.read_csv('/content/malaria_weekly.csv.gz', parse_dates=['week_start_date'])
mal_w = mal.groupby(['week_start_date','district'])['cases'].sum().reset_index()
mal_w['disease'] = 'malaria'

epi = pd.read_csv('/content/epiclim_clean.csv')
epi['week_of_outbreak'] = pd.to_datetime(epi[['year','mon','day']].rename(columns={'mon':'month'}))
epi['Cases'] = pd.to_numeric(epi['Cases'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
dengue  = epi[epi['Disease'].str.lower().str.contains('dengue', na=False)].copy()
cholera = epi[epi['Disease'].str.lower().str.contains('cholera', na=False)].copy()
dengue_w = dengue.groupby(['week_of_outbreak','district'])['Cases'].sum().reset_index()
dengue_w.columns  = ['week_start_date','district','cases']; dengue_w['disease'] = 'dengue'
cholera_w = cholera.groupby(['week_of_outbreak','district'])['Cases'].sum().reset_index()
cholera_w.columns = ['week_start_date','district','cases']; cholera_w['disease'] = 'cholera'

cases = pd.concat([mal_w, dengue_w, cholera_w], ignore_index=True)
cases['district_lower'] = cases['district'].str.lower().str.strip()
cases['node_idx'] = cases['district_lower'].map(name_to_idx)
cases = cases.dropna(subset=['node_idx'])
cases['node_idx'] = cases['node_idx'].astype(int)

weeks = sorted(cases['week_start_date'].unique())
week_to_t = {w: i for i, w in enumerate(weeks)}
T = len(weeks)

X_all = np.zeros((T, N_NODES, D), dtype=np.float32)
for _, row in cases.iterrows():
    t = week_to_t.get(row['week_start_date'])
    n = int(row['node_idx'])
    d = disease_names.index(row['disease']) if row['disease'] in disease_names else -1
    if t is not None and d >= 0:
        X_all[t, n, d] = np.log1p(float(row['cases']))

scaler = StandardScaler()
X_flat = scaler.fit_transform(X_all.reshape(-1, D))
X_all  = X_flat.reshape(T, N_NODES, D)
X_tensor = torch.FloatTensor(X_all).to(DEVICE)
print(f'Node features: {X_all.shape} (T x N x D = {T} x {N_NODES} x {D})')

In [ ]:
# ── Graph WaveNet Architecture ────────────────────────────────────────────────
# Wu et al. (2019): Adaptive Adjacency = fixed (geo) + learned (transmission)
# A_learned = softmax(ReLU(E1 @ E2.T))
# Temporal: stacked dilated causal conv1d (WaveNet-style)

class WaveNetBlock(nn.Module):
    def __init__(self, channels, dilation, kernel_size=2):
        super().__init__()
        self.filter_conv = nn.Conv1d(channels, channels, kernel_size,
                                     dilation=dilation, padding=dilation*(kernel_size-1))
        self.gate_conv   = nn.Conv1d(channels, channels, kernel_size,
                                     dilation=dilation, padding=dilation*(kernel_size-1))
        self.res_conv    = nn.Conv1d(channels, channels, 1)

    def forward(self, x):
        # x: [B, channels, T]
        f = torch.tanh(self.filter_conv(x)[..., :x.size(-1)])
        g = torch.sigmoid(self.gate_conv(x)[..., :x.size(-1)])
        out = f * g
        return self.res_conv(out) + x  # residual

class GraphWaveNet(nn.Module):
    def __init__(self, n_nodes, in_dim, out_dim, pred_len, emb_dim=10, channels=32, n_blocks=4):
        super().__init__()
        self.n_nodes  = n_nodes
        self.pred_len = pred_len
        # Adaptive adjacency
        self.E1 = nn.Embedding(n_nodes, emb_dim)
        self.E2 = nn.Embedding(n_nodes, emb_dim)
        node_ids = torch.arange(n_nodes)
        self.register_buffer('node_ids', node_ids)
        # Dilated WaveNet blocks
        self.input_proj = nn.Conv1d(in_dim, channels, 1)
        self.blocks = nn.ModuleList([
            WaveNetBlock(channels, dilation=2**i) for i in range(n_blocks)
        ])
        # Graph mixing per block
        self.graph_convs = nn.ModuleList([
            nn.Linear(channels, channels) for _ in range(n_blocks)
        ])
        self.output = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(channels, channels//2, 1),
            nn.ReLU(),
            nn.Conv1d(channels//2, out_dim * pred_len, 1),
        )

    def forward(self, x, adj_norm):
        # x: [N, D, T]  adj_norm: [N, N]
        N, D, T = x.shape
        # Learnable adjacency
        A_learn = F.softmax(F.relu(self.E1(self.node_ids) @ self.E2(self.node_ids).t()), dim=-1)
        A = (adj_norm + A_learn).clamp(0, 1)
        A = A / A.sum(dim=1, keepdim=True).clamp(min=1)

        h = self.input_proj(x)  # [N, channels, T]
        for block, gcn in zip(self.blocks, self.graph_convs):
            h = block(h)  # temporal
            h_last = h[:, :, -1]  # [N, channels]
            h_last = F.relu(gcn(A @ h_last))  # spatial mixing [N, channels]
            h[:, :, -1] = h[:, :, -1] + h_last  # inject spatial context at last step

        out = self.output(h)[:, :, -1]  # [N, D*pred_len]
        return out.view(N, D, self.pred_len)

SEQ_LEN  = 24
PRED_LEN = 4
model = GraphWaveNet(N_NODES, D, D, PRED_LEN).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
crit  = nn.HuberLoss()
print(f'Graph WaveNet params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training ───────────────────────────────────────────────────────────────────
EPOCHS = 60
cutoff = int(T * 0.85)
train_losses = []; val_losses = []

for epoch in range(EPOCHS):
    model.train()
    e_loss = 0; nb = 0
    for t in range(SEQ_LEN, cutoff - PRED_LEN, 4):
        x_in  = X_tensor[t-SEQ_LEN:t].permute(1, 2, 0)  # [N, D, SEQ]
        y_out = X_tensor[t:t+PRED_LEN].permute(1, 2, 0)  # [N, D, PRED]
        opt.zero_grad()
        pred = model(x_in, adj_norm)
        loss = crit(pred, y_out)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step(); e_loss += loss.item(); nb += 1
    avg = e_loss / max(nb, 1); train_losses.append(avg)

    model.eval()
    with torch.no_grad():
        x_v = X_tensor[cutoff-SEQ_LEN:cutoff].permute(1, 2, 0)
        y_v = X_tensor[cutoff:cutoff+PRED_LEN].permute(1, 2, 0)
        vl  = crit(model(x_v, adj_norm), y_v).item()
        val_losses.append(vl); sched.step(vl)

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Train: {avg:.4f} | Val: {vl:.4f}')

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    x_test = X_tensor[cutoff-SEQ_LEN:cutoff].permute(1, 2, 0)
    y_true = X_tensor[cutoff:cutoff+PRED_LEN].permute(1, 2, 0).cpu().numpy()
    y_pred = model(x_test, adj_norm).cpu().numpy()

yt = y_true.flatten(); yp = y_pred.flatten()
mae  = mean_absolute_error(yt, yp)
rmse = np.sqrt(mean_squared_error(yt, yp))
r2   = r2_score(yt, yp)
print('=' * 50)
print('  GRAPH WAVENET VALIDATION METRICS')
print('=' * 50)
print(f'  MAE  (norm log scale): {mae:.4f}')
print(f'  RMSE (norm log scale): {rmse:.4f}')
print(f'  R2 score:              {r2:.4f}')
print('=' * 50)
for di, dname in enumerate(disease_names):
    mae_d = mean_absolute_error(y_true[:,di,:].flatten(), y_pred[:,di,:].flatten())
    r2_d  = r2_score(y_true[:,di,:].flatten(), y_pred[:,di,:].flatten())
    print(f'  {dname:<12}: MAE={mae_d:.4f} | R2={r2_d:.4f}')

In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────────
import matplotlib; matplotlib.use('Agg')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, label='Train', color='#2563EB', lw=2)
axes[0].plot(val_losses,   label='Val',   color='#DC2626', lw=2, ls='--')
axes[0].set_title('Graph WaveNet Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Learned adjacency heatmap (top 50 districts only)
with torch.no_grad():
    A_learn = F.softmax(F.relu(model.E1(model.node_ids) @ model.E2(model.node_ids).t()), dim=-1)
    A_top = A_learn[:50, :50].cpu().numpy()
import seaborn as sns
sns.heatmap(A_top, ax=axes[1], cmap='Blues', xticklabels=False, yticklabels=False)
axes[1].set_title('Learned Adjacency (top 50 nodes)\nNon-zero = learned disease transmission links')

plt.suptitle(f'Graph WaveNet  |  Val R2={r2:.3f}', fontweight='bold')
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'graphwavenet_results.png', dpi=150)
plt.show(); print('Plots saved.')

In [ ]:
# ── Save + ensemble instructions ──────────────────────────────────────────────
import torch; from datetime import datetime
ckpt_path = DRIVE_OUTPUT + 'graphwavenet_best.pt'
torch.save({'state_dict': model.state_dict(), 'scaler': scaler,
            'adj_norm': adj_norm.cpu(), 'node_list': node_list,
            'disease_names': disease_names, 'SEQ_LEN': SEQ_LEN, 'PRED_LEN': PRED_LEN,
            'metrics': {'MAE': float(mae), 'RMSE': float(rmse), 'R2': float(r2)},
            'trained_at': datetime.now().isoformat()}, ckpt_path)
from google.colab import files
files.download(ckpt_path)
print(f'Saved: {ckpt_path}')
print(f'MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}')
print()
print('ENSEMBLE INFERENCE: load both stgnn_best.pt + graphwavenet_best.pt and average predictions.')